In [1]:
import time
import requests
import pandas as pd

from bs4 import BeautifulSoup
from datetime import date
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

print("All imports successful!")

All imports successful!


In [ ]:



# ==================================================
# 2. URL
# ==================================================

url = "https://merolagani.com/Floorsheet.aspx"


# ==================================================
# 3. SETUP CHROME
# ==================================================

chrome_options = Options()

chrome_options.add_argument(
    "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/151.0.0.0 Safari/537.36"
)

driver = webdriver.Chrome(options=chrome_options)

driver.get(url)


# ==================================================
# 4. WAIT FOR TABLE
# ==================================================

wait = WebDriverWait(driver, 30)

wait.until(
    EC.presence_of_element_located(
        (By.CSS_SELECTOR, "table.table")
    )
)

print("Website loaded successfully")


# ==================================================
# 5. FUNCTION TO EXTRACT CURRENT PAGE
# ==================================================

def extract_current_page(driver):

    # Get current HTML
    html = driver.page_source

    # Parse HTML
    soup = BeautifulSoup(html, "html.parser")

    # Find floorsheet table
    table = soup.find(
        "table",
        class_="table table-bordered table-striped table-hover sortable"
    )

    if table is None:
        print("Table not found")
        return pd.DataFrame()

    # Find table body
    tbody = table.find("tbody")

    if tbody is None:
        print("Table body not found")
        return pd.DataFrame()

    # Find rows
    rows = tbody.find_all("tr")

    data = []

    for row in rows:

        cells = row.find_all("td")

        if len(cells) == 8:

            data.append({
                "transaction_no": cells[1].get_text(strip=True),
                "symbol": cells[2].get_text(strip=True),
                "buyer": cells[3].get_text(strip=True),
                "seller": cells[4].get_text(strip=True),
                "quantity": cells[5].get_text(strip=True),
                "rate": cells[6].get_text(strip=True),
                "amount": cells[7].get_text(strip=True)
            })

    return pd.DataFrame(data)


# ==================================================
# 6. SCRAPE ALL PAGES
#    NO FIXED TOTAL PAGE
# ==================================================

all_pages = []

page_number = 1

while True:

    print(
        f"\nScraping page {page_number}..."
    )

    # ----------------------------------------------
    # Extract current page
    # ----------------------------------------------

    df_page = extract_current_page(driver)

    print(
        f"Rows extracted: {len(df_page)}"
    )

    # If no data found, stop
    if df_page.empty:

        print(
            "No data found on current page."
        )

        break

    # Add page data
    all_pages.append(df_page)


    # ----------------------------------------------
    # Get current first transaction
    # ----------------------------------------------

    try:

        old_transaction = driver.find_element(
            By.CSS_SELECTOR,
            "table.table tbody tr td:nth-child(2)"
        ).text.strip()

    except Exception:

        print(
            "Could not find current transaction."
        )

        break


    # ----------------------------------------------
    # Find Next Page button
    # ----------------------------------------------

    next_buttons = driver.find_elements(
        By.XPATH,
        "//a[contains(@title, 'Next Page')]"
    )


    # ----------------------------------------------
    # Check if Next button exists
    # ----------------------------------------------

    if not next_buttons:

        print(
            "\nNext Page button not found."
        )

        print(
            "Reached last page."
        )

        break


    # ----------------------------------------------
    # Use last Next button
    # ----------------------------------------------

    next_button = next_buttons[-1]


    # ----------------------------------------------
    # Check if Next button is disabled
    # ----------------------------------------------

    try:

        disabled = next_button.get_attribute("disabled")
        class_attribute = next_button.get_attribute("class")

        if (
            disabled is not None
            or (
                class_attribute
                and "disabled" in class_attribute.lower()
            )
        ):

            print(
                "\nNext Page button is disabled."
            )

            print(
                "Reached last page."
            )

            break

    except Exception:

        pass


    # ----------------------------------------------
    # Click Next Page
    # ----------------------------------------------

    try:

        driver.execute_script(
            "arguments[0].click();",
            next_button
        )

    except Exception as e:

        print(
            f"Could not click Next Page: {e}"
        )

        break


    # ----------------------------------------------
    # Wait for page change
    # ----------------------------------------------

    try:

        wait.until(
            lambda d:
            d.find_element(
                By.CSS_SELECTOR,
                "table.table tbody tr td:nth-child(2)"
            ).text.strip()
            != old_transaction
        )

    except Exception:

        print(
            "Could not detect page change."
        )

        # Give website extra time
        time.sleep(2)


    # ----------------------------------------------
    # Increase page number
    # ----------------------------------------------

    page_number += 1


# ==================================================
# 7. CLOSE BROWSER
# ==================================================

driver.quit()


# ==================================================
# 8. COMBINE ALL PAGES
# ==================================================

if all_pages:

    df = pd.concat(
        all_pages,
        ignore_index=True
    )

else:

    df = pd.DataFrame()


# ==================================================
# 9. CONVERT NUMERIC COLUMNS
# ==================================================

if not df.empty:

    df["quantity"] = pd.to_numeric(
        df["quantity"],
        errors="coerce"
    )

    df["rate"] = pd.to_numeric(
        df["rate"],
        errors="coerce"
    )

    df["amount"] = pd.to_numeric(
        df["amount"]
        .astype(str)
        .str.replace(",", "", regex=False),
        errors="coerce"
    )


# ==================================================
# 10. REMOVE DUPLICATES
# ==================================================

if not df.empty:

    df = df.drop_duplicates(
        subset=["transaction_no"]
    ).reset_index(drop=True)


# ==================================================
# 11. RESULT
# ==================================================

print("\n" + "=" * 60)
print("SCRAPING COMPLETED")
print("=" * 60)

print(
    "Total pages scraped:",
    page_number
)

print(
    "Total rows:",
    len(df)
)

print(
    "Total columns:",
    len(df.columns)
)

print("\nDataFrame:")
print(df.head())

print("\nData types:")
print(df.dtypes)

Website loaded successfully

Scraping page 1...
Rows extracted: 500
Could not detect page change.

Scraping page 2...
Rows extracted: 500

Scraping page 3...
Rows extracted: 500

Scraping page 4...
Rows extracted: 500

Scraping page 5...
Rows extracted: 500

Scraping page 6...
Rows extracted: 500

Scraping page 7...
Rows extracted: 500

Scraping page 8...
Rows extracted: 500

Scraping page 9...
Rows extracted: 500

Scraping page 10...
Rows extracted: 500

Scraping page 11...
Rows extracted: 500

Scraping page 12...
Rows extracted: 500

Scraping page 13...
Rows extracted: 500

Scraping page 14...
Rows extracted: 500

Scraping page 15...
Rows extracted: 500

Scraping page 16...
Rows extracted: 500

Scraping page 17...
Rows extracted: 500

Scraping page 18...
Rows extracted: 500

Scraping page 19...
Rows extracted: 500

Scraping page 20...
Rows extracted: 500

Scraping page 21...
Rows extracted: 500

Scraping page 22...
Rows extracted: 500

Scraping page 23...
Rows extracted: 500

Scrapin

In [9]:
df.head(10)

,transaction_no,symbol,buyer,seller,quantity,rate,amount
0,2026081805010274,AHL,40,10,25.0,408.0,10200.0
1,2026081805010273,AHL,11,10,50.0,408.0,20400.0
2,2026081805010272,AHL,64,10,105.0,408.0,42840.0
3,2026081805010271,AHL,89,10,70.0,408.1,28567.0
4,2026081805010270,AHL,19,10,60.0,408.2,24492.0
5,2026081805010269,AHL,26,10,30.0,408.2,12246.0
6,2026081805010268,AHL,38,10,58.0,408.2,23675.6
7,2026081805010336,BEDC,41,41,10.0,313.5,3135.0
8,2026081805010335,BEDC,41,1,30.0,313.0,9390.0
9,2026081805010121,BEDC,11,1,10.0,313.0,3130.0


In [10]:
# Sector-wise script lists
Bank = ["NABIL", "NIMB", "SCB", "HBL", "SBI", "EBL", "NICA", "MBL", "LSL", "KBL",
        "SBL", "SANIMA", "NMB", "PRVU", "GBIME", "CZBIL", "PCBL", "ADBL", "NBL"]

manufacturing = ["BNL", "NLO", "BNT", "UNL", "HDL", "SHIVM", "GCIL", "SONA",
                 "SARBTM", "OMPL", "SAGAR", "SAIL", "SYPNL", "RSML", "PCIL", "SOPL", "ECL"]

hotel_and_tourism = ["SHL", "TRH", "OHL", "CGH", "KDL", "CITY", "BANDIPUR", "HFIN"]

other = ["NTC", "NRIC", "NRM", "MKCL", "NWCL", "HRL", "PURE", "TTL"]

hydropower = [
    "NHPC", "BPCL", "CHCL", "AHPC", "SHPC", "RIDI", "BARUN", "API",
    "NGPL", "KKHC", "DHPL", "AKPL", "SPDL", "UMHL", "CHL", "HPPL",
    "NHDL", "RADHI", "PMHPL", "KPCL", "AKJCL", "JOSHI", "UPPER", "GHL",
    "UPCL", "MHNL", "PPCL", "HURJA", "UNHPL", "RHPL", "SJCL", "HDHPC",
    "LEC", "SSHL", "MEN", "UMRH", "GLH", "SHEL", "RURU", "MKJC",
    "SAHAS", "TPC", "SPC", "NYADI", "MBJC", "BNHC", "GVL", "BHL",
    "RFPL", "DORDI", "BHDC", "HHL", "UHEWA", "SGHC", "MHL", "USHEC",
    "RHGCL", "SPHL", "PPL", "SIKLES", "EHPL", "PHCL", "BHPL", "SMHL",
    "SPL", "SMH", "MKHC", "AHL", "TAMOR", "MHCL", "SMJC", "MAKAR",
    "MKHL", "DOLTI", "BEDC", "MCHL", "IHL", "MEL", "RAWA", "USHL",
    "TSHL", "KBSH", "MEHL", "ULHC", "MANDU", "BGWT", "MSHL", "MMKJL",
    "TVCL", "VLUCL", "CKHL", "SANVI", "BHCL", "HIMSTAR", "MABEL", "DHEL",
    "BUNGAL", "SOHL", "BJHL", "SKHL", "RLEL", "SKHEL", "SIPD", "KHPL",
    "APHL", "YMHL", "TPKHL", "SNORL", "SGHL", "KAHL", "MEPDL"
]

trading = ["STC", "BBC"]

non_life_insurance = [
    "NICL", "RBCL", "HEI", "UAIL", "SPIL", "NIL", "PRIN",
    "SALICO", "IGI", "SICL", "NLG", "SGIC", "NMIC"
]

development_bank = [
    "NABBC", "EDBL", "LBBL", "MDB", "MLBL", "GBBL", "JBBL", "CORBL",
    "KSBBL", "SADBL", "SHINE", "MNBBL", "SINDU", "GRDBL", "SAPDBL", "SABBL"
]

finance = [
    "NFS", "GUFL", "BFC", "GFCL", "SIFC", "CFCL", "JFL",
    "GMFIL", "ICFC", "PROFL", "MPFL", "MFIL", "RLFL"
]

microfinance = [
    "NUBL", "CBBL", "DDBL", "SWBBL", "NMLBBL", "FMDBL", "SLBBL", "SKBBL",
    "GBLBS", "KMCDB", "MLBBL", "LLBS", "VLBS", "HLBSL", "MATRI", "JSLBB",
    "NMBMF", "GILB", "SWMF", "MERO", "NMFBS", "RSDC", "FOWAD", "SMATA",
    "MSLB", "SMB", "USLB", "WNLB", "NADEP", "ACLBSL", "SLBSL", "ALBSL",
    "GMFBS", "GLBSL", "SMFBS", "ILBS", "NICLBSL", "SMPDA", "MLBSL", "JBLB",
    "MLBS", "NESDO", "ULBSL", "CYCL", "AVYAN", "DLBS", "SHLB", "UNLB",
    "ANLB", "SWASTIK"
]

life_insurance = [
    "NLICL", "NLIC", "LICN", "ALICL", "HLI", "SJLIC", "PMLI",
    "SRLI", "ILI", "RNLI", "SNLI", "CLI", "GMLI", "CREST"
]

investment = [
    "CIT", "HIDCL", "NRN", "NIFRA", "CHDC", "ENL", "HATHY"
]
mutual_fund = [
    "SEF", "NBF2", "SIGS2", "NICBF", "NMB50", "SFMF", "LUK", "SLCF",
    "KEF", "SBCF", "PSF", "NIBSF2", "NICSF", "RMF1", "MMF1", "NBF3",
    "NICFC", "KDBY", "GIBF1", "NSIF2", "NIBLGF", "SAGF", "SFEF", "PRSF",
    "RMF2", "SIGS3", "C30MF", "LVF2", "H8020", "NICGF2", "KSY", "NIBLSTF",
    "MNMF1", "GSY", "NMBHF2", "MBLEF", "RSY", "GBIMESY2", "HLICF", "RBBF40",
    "CSY", "NSY", "SEF2", "SAEF2", "LSH12", "RSY2"
]

preference_share = [
    "NABILPNP",
    "KSBBLPNP",
    "SBLPNP",
    "NMBPNP",
    "SANIMAPNP",
    "MBLPNP"
]

# Create sector mapping
sector_mapping = {}

for script in Bank:
    sector_mapping[script] = "Banking"

for script in manufacturing:
    sector_mapping[script] = "Manufacturing"

for script in hotel_and_tourism:
    sector_mapping[script] = "Hotel and Tourism"

for script in other:
    sector_mapping[script] = "Other"

for script in hydropower:
    sector_mapping[script] = "Hydropower"

for script in trading:
    sector_mapping[script] = "Trading"

for script in non_life_insurance:
    sector_mapping[script] = "Non-Life Insurance"

for script in development_bank:
    sector_mapping[script] = "Development Bank"

for script in finance:
    sector_mapping[script] = "Finance"

for script in microfinance:
    sector_mapping[script] = "Microfinance"

for script in life_insurance:
    sector_mapping[script] = "Life Insurance"

for script in investment:
    sector_mapping[script] = "Investment"

for script in mutual_fund:
    sector_mapping[script] = "Mutual Fund"

for script in preference_share:
    sector_mapping[script] = "Preference Share"

# Create sector column
df["sector"] = df["symbol"].map(sector_mapping)

In [11]:
df["date"] = pd.Timestamp.today().date()

In [12]:
from pathlib import Path
from datetime import date

# Go from src/scraper -> project root
project_root = Path.cwd().parents[1]

# data/raw
raw_folder = project_root / "data" / "raw"/"floorsheet_data"

# Create folder if it doesn't exist
raw_folder.mkdir(parents=True, exist_ok=True)

# Filename
today_date = date.today().strftime("%Y-%m-%d")
file_path = raw_folder / f"{today_date}-stock_data.csv"

# Save
df.to_csv(file_path, index=False)

print(f"Saved successfully: {file_path}")

Saved successfully: e:\nepse-market-report\data\raw\floorsheet_data\2026-08-18-stock_data.csv


In [14]:
# for combining all the data in one file, we can use the following code:
# ============================================================
# 1. COPY SCRAPED DATA
# ============================================================

df2 = df.copy()


# ============================================================
# 2. ADD TODAY'S DATE
# ============================================================




# ============================================================
# 3. GO FROM src/scraper -> PROJECT ROOT
# ============================================================

project_root = Path.cwd().parents[1]


# ============================================================
# 4. DATA/RAW/HISTORIC_STOCK_DATA FOLDER
# ============================================================

raw_folder = (
    project_root
    / "data"
    / "raw"
    / "historic_stock_data"
)

raw_folder.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 5. CSV FILE PATH
# ============================================================

csv_file = (
    raw_folder
    / "historic_floorsheet_data.csv"
)


# ============================================================
# 6. APPEND NEW DATA TO OLD DATA
# ============================================================

if csv_file.exists():

    # Read existing historical data
    old_df = pd.read_csv(csv_file)

    # Combine old + today's data
    historic_df = pd.concat(
        [old_df, df2],
        ignore_index=True
    )

else:

    # First time creating the file
    historic_df = df2


# ============================================================
# 7. SAVE HISTORICAL DATA
# ============================================================

historic_df.to_csv(
    csv_file,
    index=False
)


# ============================================================
# 8. INFORMATION
# ============================================================

print("Historical data saved successfully!")
print("File:", csv_file)
print("Shape:", historic_df.shape)


C:\Users\DELL\AppData\Local\Temp\ipykernel_25020\2406604608.py:57: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  old_df = pd.read_csv(csv_file)


Historical data saved successfully!
File: e:\nepse-market-report\data\raw\historic_stock_data\historic_floorsheet_data.csv
Shape: (248369, 10)


In [1]:
import re
import pandas as pd
from snowflake.connector.pandas_tools import write_pandas


# ============================================================
# 1. COPY df2
# ============================================================

df = df2.copy()


# ============================================================
# 2. CLEAN COLUMN NAMES
# ============================================================

df.columns = [
    re.sub(
        r"[^A-Z0-9_]",
        "_",
        str(col).strip().upper()
    ).strip("_")
    for col in df.columns
]


# ============================================================
# 3. EXPECTED SNOWFLAKE COLUMNS
# ============================================================

expected_columns = [
    "TRANSACTION_NO",
    "SYMBOL",
    "BUYER",
    "SELLER",
    "QUANTITY",
    "RATE",
    "AMOUNT",
    "SECTOR",
    "DATE"
]


# ============================================================
# 4. CHECK COLUMNS
# ============================================================

missing_columns = [
    col for col in expected_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns in df2: {missing_columns}"
    )


# Keep only required columns
df = df[expected_columns]


# ============================================================
# 5. CONVERT DATA TYPES
# ============================================================

df["TRANSACTION_NO"] = pd.to_numeric(
    df["TRANSACTION_NO"],
    errors="coerce"
)

df["QUANTITY"] = pd.to_numeric(
    df["QUANTITY"],
    errors="coerce"
)

df["RATE"] = pd.to_numeric(
    df["RATE"],
    errors="coerce"
)

df["AMOUNT"] = pd.to_numeric(
    df["AMOUNT"],
    errors="coerce"
)

df["DATE"] = pd.to_datetime(
    df["DATE"],
    errors="coerce"
).dt.date


# ============================================================
# 6. RESET INDEX
# ============================================================

df.reset_index(drop=True, inplace=True)


# ============================================================
# 7. CHECK DATA
# ============================================================

print("========================================")
print("FLOORSHEET DATA")
print("========================================")

print("Total rows:", len(df))
print("Total columns:", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


# ============================================================
# 8. INSERT ALL ROWS INTO SNOWFLAKE
# ============================================================

success, nchunks, nrows, output = write_pandas(
    conn=conn,
    df=df,
    database="NEPSE_DB",
    schema="NEPSE_SCHEMA",
    table_name="FLOORSHEET",
    chunk_size=10000,
    auto_create_table=False,
    overwrite=False
)


# ============================================================
# 9. RESULT
# ============================================================

print("\n========================================")
print("FLOORSHEET UPLOAD RESULT")
print("========================================")

print("Success       :", success)
print("Chunks        :", nchunks)
print("Rows inserted :", nrows)

if success:
    print("\nAll floor-sheet records uploaded successfully!")
else:
    print("\nUpload failed.")
    print(output)

NameError: name 'df2' is not defined